# 00 — Inspection and loading

Raw-format reconnaissance only. No filtering, no corrections, no statistics: those start in 01/02/03.
Output = tidy dataframes in `outputs/interim/` plus a verified inventory of what is standard and what is sample.

**What `Information regarding your files.docx` actually says** (read before touching the data):

* Microprobe glass: 3 files (1 oxide + 2 conditions/standards). Olivine: 5 files (oxide, element/mole, Fo-Fa ratios, 2 conditions).
* Sample naming: the **last** number in a name is the measurement number, so `1982-CS2-L3A_1..9` are 9 spots on one sample. Glass = 9 spots per sample; olivine = irregular number of spots per grain, transects prefixed `Line`.
* EPMA standards: `SC-Ol` for olivine, `VG2`, `VGA99` and `Scap` for glass. Data is already matrix-corrected in the machine — the job is to check whether it is right. Scapolite has a different matrix, so many elements will look off; it is there because a few elements are calibrated on it.
* LA-ICP-MS: **KL2 is the calibration standard**. Everything not starting with `MC` is a standard run as an unknown, for accuracy/precision checks. NIST was measured for overall machine performance only. Run schedule: NIST → 3× KL2 → ~20 samples → 3× KL2 → ~20 samples → … → 3× KL2 → NIST.
* LA samples are melt inclusions, mostly single analyses; duplicates are flagged with A/B/C suffixes (A sometimes missing).

The .docx makes **no mention of split sets** — the nine data files here are the complete exercise: one glass session, one olivine session, one LA-ICP-MS session.

In [12]:
# --- config ---
import warnings; warnings.filterwarnings("ignore", message="Unknown extension is not supported")
from pathlib import Path
import sys, re, numpy as np, pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
INTERIM = ROOT / "outputs" / "interim"; INTERIM.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

GLASS = DATA / "bgls_20122016"           # + _all / _oxide / _std-cnd .txt
OL    = DATA / "ol_01032017"             # + _all / _oxide / _mole / _ratios / _std-cnd .txt
LA    = next(DATA.glob("170919s_9_global_tephra*.csv"))   # filename contains a space in the original
STD   = DATA / "Standards.xlsx"

pd.set_option("display.width", 160, "display.max_columns", 40)

## 1. Raw look — delimiter, header depth, encoding

`head` first, parser second.

In [13]:
for p in [GLASS.with_name(GLASS.name + "_oxide.txt"), OL.with_name(OL.name + "_oxide.txt"),
          GLASS.with_name(GLASS.name + "_all.txt"), LA]:
    raw = p.read_bytes()[:900]
    print("===== {}   CRLF={}  tabs={}  commas={}".format(p.name, b"\r\n" in raw, raw.count(b"\t"), raw.count(b",")))
    print(raw.decode("utf-8", "replace")[:600], "\n")

===== bgls_20122016_oxide.txt   CRLF=False  tabs=83  commas=0
 
Mass percent  	      Group : bas-Glas      	  Sample : 201216_Fr       	 Page 	  1 	 
 
   No. 	   Na2O  	   SO3   	   FeO   	   SiO2  	   Cl    	   F     	   K2O   	   MnO   	   Al2O3 	   P2O5  	   MgO   	   CaO   	   TiO2  	  Total  	Comment  
    1  	  2.8000 	  0.0408 	 13.6000 	 51.1600 	  0.0237 	  0.0126 	  0.8138 	  0.1786 	 12.3000 	  0.3881 	  5.1300 	  9.2700 	  4.0800 	 99.7976 	VGA99_ 
    2  	  2.7000 	  0.0232 	 13.4600 	 51.2400 	  0.0300 	  0.0791 	  0.8142 	  0.2468 	 12.4800 	  0.3543 	  5.2000 	  9.3700 	  4.0600 	100.0575 	VGA99_ 
    3  	  2.6900 	  0.3666 	 11.7600 	 5 

===== ol_01032017_oxide.txt   CRLF=True  tabs=75  commas=0
 
Mass percent  	      Group : Olivine       	  Sample : 010317Fr        	 Page 	  1 	 
 
   No. 	   MgO   	   MnO   	   FeO   	   SiO2  	   CaO   	   Cr2O3 	   NiO   	   Al2O3 	  Total  	Comment  
    1  	 49.7800 	  0.1478 	  9.5600 	 39.7200 	  0.1024 	  0.0311 	  0.3672 	

Three distinct formats, all plain ASCII (olivine files CRLF, glass LF):

| File | Format |
|---|---|
| `_oxide`, `_mole`, `_ratios` | tab-separated, 3 junk lines then 1 header row, `No.` … `Total`, `Comment` last; **footer block** `Minimum/Maximum/Average/Sigma/"No. of data"` that must be dropped |
| `_all` | not tabular — one whitespace-aligned block per analysis: stage coords, kV, beam diameter, date/time, current, then net cps / background / S.D.(%) / **D.L.(ppm)** per element, then ZAF factors and element/oxide mass% |
| `_std-cnd` | tab-separated sections: WDS spectrometer setup, peak/background **count times**, `Standard Data` (which standard calibrates which oxide), standard intensities with their measurement dates |
| `*.csv` (GLITTER) | 7 stacked blocks, each `Element` rows × analysis columns: MDL-filtered ppm, unfiltered ppm, 1σ error, MDL, chondrite-normalised, raw cps (bg not subtracted), raw cps (bg subtracted). Below-MDL values are written as `<value` — **strings, not numbers** |

`_all` is the only source of detection limits and per-analysis S.D.(%), and the only place with a timestamp per analysis — that timestamp is what orders the drift plots in 01/02, not the `No.` column.

In [14]:
from geohelpers import load_jeol_table, load_jeol_all, load_std_cnd, load_glitter, parse_id

gl_ox  = load_jeol_table(GLASS.with_name(GLASS.name + "_oxide.txt"))
ol_ox  = load_jeol_table(OL.with_name(OL.name + "_oxide.txt"))
ol_mol = load_jeol_table(OL.with_name(OL.name + "_mole.txt"))
ol_rat = load_jeol_table(OL.with_name(OL.name + "_ratios.txt"))

gl_meta, gl_int = load_jeol_all(GLASS.with_name(GLASS.name + "_all.txt"))
ol_meta, ol_int = load_jeol_all(OL.with_name(OL.name + "_all.txt"))

gl_cal, gl_cnd = load_std_cnd(GLASS.with_name(GLASS.name + "_std-cnd.txt"))
ol_cal, ol_cnd = load_std_cnd(OL.with_name(OL.name + "_std-cnd.txt"))

la = load_glitter(LA)   # {block name: (values, below-MDL mask)}

for n, d in [("glass oxide", gl_ox), ("ol oxide", ol_ox), ("ol mole", ol_mol), ("ol ratios", ol_rat),
             ("glass meta", gl_meta), ("ol meta", ol_meta)]:
    print(f"{n:12s} {d.shape}")
print({k: v[0].shape for k, v in la.items()})

glass oxide  (105, 16)
ol oxide     (128, 11)
ol mole      (128, 11)
ol ratios    (128, 5)
glass meta   (105, 10)
ol meta      (128, 10)
{'Trace Element Concentrations MDL filtered': (85, 41), 'Trace Element Concentrations, Not filtered for MDL': (85, 41), '1 sigma error': (85, 41), 'Minimum detection limits (99% confidence)': (85, 41), 'Trace element concentrations normalised to chondrite': (85, 41), 'Mean Raw CPS background NOT subtracted': (85, 41), 'Mean Raw CPS background subtracted': (85, 41)}


## 2. Row counts against the .docx

The JEOL files print their own `"No. of data"` in the footer; the `_all` block count and the oxide table must agree with it, otherwise the parser is silently dropping analyses.

In [15]:
def footer_n(p):
    return int(re.search(r'"No\. of data"\s+(\d+)', p.read_text()).group(1))

checks = {
    "glass: oxide rows":   len(gl_ox),
    "glass: footer n":     footer_n(GLASS.with_name(GLASS.name + "_oxide.txt")),
    "glass: _all blocks":  len(gl_meta),
    "olivine: oxide rows": len(ol_ox),
    "olivine: footer n":   footer_n(OL.with_name(OL.name + "_oxide.txt")),
    "olivine: _all blocks": len(ol_meta),
    "olivine: mole/ratios": (len(ol_mol), len(ol_rat)),
    "LA: analyses":        la["Trace Element Concentrations MDL filtered"][0].shape[0],
    "LA: elements":        la["Trace Element Concentrations MDL filtered"][0].shape[1],
}
for k, v in checks.items(): print(f"{k:22s} {v}")

assert len(gl_ox) == len(gl_meta) == 105
assert len(ol_ox) == len(ol_meta) == len(ol_mol) == len(ol_rat) == 128
assert (gl_ox["No."].values == gl_meta["No."].values).all()
assert (ol_ox["No."].values == ol_meta["No."].values).all()
assert (gl_ox["Comment"].values == gl_meta["Comment"].values).all()   # oxide table and _all describe the same analyses
print("\nrow counts consistent across files")

glass: oxide rows      105
glass: footer n        105
glass: _all blocks     105
olivine: oxide rows    128
olivine: footer n      128
olivine: _all blocks   128
olivine: mole/ratios   (128, 128)
LA: analyses           85
LA: elements           41

row counts consistent across files


## 3. Material classification

Split on the `Comment` field (EPMA) and the analysis label (LA). Nothing is dropped here — every analysis gets a label and is kept.

In [16]:
GL_STD = {"VGA99_": "VGA99", "VG-2_": "VG2", "Scap": "Scapolite", "KE3_": "KE3"}

def tag_epma(df, std_map):
    out = df.copy()
    out["material"] = out["Comment"].map(std_map)
    out["is_std"]   = out["material"].notna()
    out["sample"], out["spot"] = zip(*out["Comment"].map(parse_id))
    out.loc[out["is_std"], ["sample", "spot"]] = [np.nan, np.nan]
    out["material"] = out["material"].fillna(out["sample"])
    return out

gl_ox = tag_epma(gl_ox, GL_STD)
ol_ox = tag_epma(ol_ox, {"SC-Olivine": "SC-Ol"})
for d in (ol_mol, ol_rat):
    d[["material", "is_std", "sample", "spot"]] = ol_ox[["material", "is_std", "sample", "spot"]]

print("GLASS — standards:", gl_ox.loc[gl_ox.is_std, "material"].value_counts().to_dict())
print("GLASS — samples (spots each):")
print(gl_ox.loc[~gl_ox.is_std, "sample"].value_counts().sort_index().to_string())
print("\nOLIVINE — standards:", ol_ox.loc[ol_ox.is_std, "material"].value_counts().to_dict())
print("OLIVINE — grains:", ol_ox.loc[~ol_ox.is_std, "sample"].nunique(),
      "| spots per grain:", ol_ox.loc[~ol_ox.is_std, "sample"].value_counts().value_counts().to_dict())
print("OLIVINE — transect ('Line') analyses:", ol_ox["Comment"].str.contains("Line", case=False).sum())

GLASS — standards: {'VGA99': 6, 'VG2': 6, 'KE3': 6, 'Scapolite': 6}
GLASS — samples (spots each):
sample
1982-CS2-L3A          9
1982-CS2-L5B          9
UnknA-BatokeCS1-3     9
UnknA-C21             9
UnknA-C22-PS1-L2      9
UnknA-C22-PS1-L3      9
UnknA-Flow1-1        18
UnknA-Flow1-2         9

OLIVINE — standards: {'SC-Ol': 15}
OLIVINE — grains: 37 | spots per grain: {3: 35, 4: 2}
OLIVINE — transect ('Line') analyses: 0


In [17]:
# LA: label = 3-digit run number + material. Everything not starting with MC is a standard (per the .docx).
conc, bdl = la["Trace Element Concentrations MDL filtered"]
la_idx = pd.DataFrame({"label": conc.index})
la_idx["run"] = la_idx["label"].str.extract(r"^(\d+)").astype(int)
la_idx["name"] = la_idx["label"].str.replace(r"^\d+", "", regex=True)
la_idx["is_sample"] = la_idx["name"].str.startswith("MC")
la_idx["material"] = np.where(la_idx.is_sample, la_idx["name"].str.replace(r"([A-C])$", "", regex=True),
                              la_idx["name"].str.replace(r"\d*$", "", regex=True))
la_idx.loc[~la_idx.is_sample, "material"] = la_idx.loc[~la_idx.is_sample, "name"]
la_idx["role"] = np.where(la_idx.is_sample, "sample",
                 np.where(la_idx.name.str.startswith("KL2"), "calibration std (KL2)",
                 np.where(la_idx.name.str.startswith("N612"), "NIST — machine check", "control std")))

print(la_idx.groupby(["role", "material"]).size().to_string())
print("\nrun numbers:", la_idx.run.min(), "→", la_idx.run.max(), "| missing:",
      sorted(set(range(la_idx.run.min(), la_idx.run.max() + 1)) - set(la_idx.run)))
print("KL2 bracket positions:", la_idx.loc[la_idx.role.str.startswith("calibration"), "run"].tolist())

role                   material  
NIST — machine check   N612           7
calibration std (KL2)  KL2           17
control std            BCR2G          4
                       BM90           2
                       GOR128         4
                       GOR132         4
sample                 MC22-1-16      1
                       MC22-1-20      1
                       MC22-1-21      1
                       MC22-1-26      2
                       MC22-1-27      1
                       MC22-2-1       1
                       MC22-2-23      1
                       MC22-2-24      1
                       MC22-2-27      1
                       MC22-2-28      1
                       MC22-2-29      2
                       MC22-2-30      1
                       MC22-2-31      1
                       MC22-2-32      1
                       MC22-2-35      1
                       MC22-2-40      1
                       MC22-2-5       1
                       MC22-2-8       1
      

**Inventory (verified, not assumed):**

* **Glass, 20/12/2016, 16:18–22:19 — 105 analyses.** Standards 6× each of VGA99, VG-2, Scapolite and **KE3** (24 analyses; the .docx lists only the first three, KE3 is the F standard and is present). Samples: 8 names, 81 spots — seven with the 9 spots the .docx promises, but **`UnknA-Flow1-1` carries 18**, i.e. the 9-spot sequence run twice. Treat it as two populations in 02 until the duplicate labels are shown to agree.
* **Olivine, 01–02/03/2017 — 128 analyses.** 15× SC-Olivine (the .docx says 3× — it is 15; check the timestamps before assuming they are session brackets), 37 melt-inclusion host grains with 3 spots each (two grains with 4). **No `Line` transects in this file** despite the .docx mentioning them.
* **LA-ICP-MS, 19/09/2017 — 85 analyses, 41 elements** (Li7…U238). Run numbers 1–86 with **033 missing** (deleted in GLITTER — one line in the methods). KL2 ×17, NIST612 ×7 (start and end), controls BCR-2G ×4, GOR128-G ×4, GOR132-G ×4, BM90/21-G ×2; 47 melt-inclusion analyses (`MC22-*`, `MC59-*`, `MC82-*`, `MC99U-*`), a few with B duplicates.
* KL2 brackets fall at runs 4–6, 24–26, 44–46, 64–67 and 81–83, which is the batch structure the .docx describes and the natural basis for a batch-wise drift correction.

## 4. Analytical conditions and calibration assignment

Straight from `_std-cnd` and `_all` — this is the raw material for the methodology section, so it is transcribed, not remembered.

In [18]:
for name, cal, cnd, meta in [("GLASS", gl_cal, gl_cnd, gl_meta), ("OLIVINE", ol_cal, ol_cnd, ol_meta)]:
    print(f"===== {name}")
    print(f"  {meta.kV.unique()} kV | beam dia {meta.dia.unique()} µm | current "
          f"{meta.curr_A.min():.2e}–{meta.curr_A.max():.2e} A | {meta.datetime.min()} → {meta.datetime.max()}")
    print(cal.merge(cnd, left_on=cal.oxide.str.extract(r'^([A-Z][a-z]?)')[0], right_on="element",
                    how="left")[["oxide", "standard", "mass_pct", "peak_s", "bg_s"]].to_string(index=False))
    print()

===== GLASS
  [15.] kV | beam dia [5.] µm | current 1.00e-08–1.01e-08 A | 2016-12-20 16:18:00 → 2016-12-20 22:19:00
oxide       standard  mass_pct  peak_s  bg_s
 Na2O     VGA99-15KV    2.6555    10.0  10.0
  SO3 Scapolite_bgls    1.3235    30.0  15.0
  FeO     VGA99-15KV   13.3022    20.0  10.0
 SiO2     VGA99-15KV   50.9342    20.0  10.0
   Cl Scapolite_bgls    1.4300    30.0  15.0
    F            KE3    0.4700    30.0  15.0
  K2O     VGA99-15KV    0.8191    30.0  15.0
  MnO      Rhodonite   43.4885    30.0  15.0
Al2O3     VGA99-15KV   12.4899    20.0  10.0
 P2O5     VGA99-15KV    0.3896    20.0  10.0
  MgO     VGA99-15KV    5.0740    20.0  10.0
  CaO     VGA99-15KV    9.3047    20.0  10.0
 TiO2     VGA99-15KV    4.0534    20.0  10.0

===== OLIVINE
  [15.] kV | beam dia [0.] µm | current 9.82e-08–9.87e-08 A | 2017-03-02 02:12:00 → 2017-03-02 09:23:00
oxide     standard  mass_pct  peak_s  bg_s
  MgO   SC-Olivine   49.4299    20.0  10.0
  MnO    Rhodonite   43.4885    60.0  30.0
  FeO 

**Conditions** — glass: 15 kV, 5 µm defocused beam, ~10 nA, 10–30 s on peak / 10–15 s background, 20 Dec 2016 16:18–22:19 (6 h session).
Olivine: 15 kV, **focused beam (`Probe Dia. : 0`)**, ~100 nA, 20–60 s on peak / 10–30 s background, 2 Mar 2017 02:12–09:23 — note the analyses run into the early hours of 2 March even though the session is labelled 01/03/2017.
The order-of-magnitude higher current and the longer counts on Ca, Mn, Ni and Al in the olivine session are why those minor oxides are measurable there at all.

**Calibration standards actually used** (this contradicts the .docx and matters for 01/02):

* Glass: **VGA99** calibrates Na₂O, FeO, SiO₂, K₂O, Al₂O₃, P₂O₅, MgO, CaO, TiO₂ — not VG2, which the .docx claims. Scapolite calibrates SO₃ and Cl, KE3 calibrates F, Rhodonite calibrates MnO. So VG-2 is a pure *secondary* standard here and is the cleanest accuracy check available; VGA99 is circular for the nine oxides it calibrates (its “accuracy” only tests reproducibility of the standard itself).
* Olivine: SC-Olivine calibrates MgO, FeO, SiO₂; Wollastonite CaO, Rhodonite MnO, and pure Cr₂O₃ / NiO / Corundum the minor oxides. SC-Ol is likewise circular for its three majors — accuracy for CaO, MnO, NiO, Cr₂O₃ and Al₂O₃ is the meaningful test, and every one of those is calibrated on a matrix far from olivine.

Both points belong in the methodology: the exercise asks “is the data correct”, and the honest answer depends on which oxide is calibrated on what.

## 5. LA-ICP-MS internal standard

GLITTER normalises to one element held constant per material. Identify it by finding the element whose value is *identical* across repeat analyses of the same material — a computed quantity, not a measured one.

In [19]:
mat = la_idx.set_index("label").loc[conc.index, "material"]
rep = conc.loc[mat.isin(["KL2", "N612", "GOR128", "GOR132"]).values]
g = rep.groupby(mat[rep.index].values)
spread = (g.std() / g.mean()).median()          # median RSD across the repeated materials
print("smallest within-material relative spread (= the normalising element):")
print(spread.sort_values().head(4).to_string(float_format="%.2e"))

ca = pd.DataFrame({"Ca43_ppm": conc["Ca43"], "material": mat})
print("\nCa43 per standard material:")
print(ca[~la_idx.set_index('label').loc[conc.index, 'is_sample'].values]
      .groupby("material")["Ca43_ppm"].agg(["size", "nunique", "mean"]).to_string())
print("\nBCR-2G analyses:"); print(ca.loc[mat == 'BCR2G', 'Ca43_ppm'].to_string())

smallest within-material relative spread (= the normalising element):
Ca43    6.00e-08
Ti47    2.07e-02
Sr88    2.55e-02
Pr141   2.59e-02

Ca43 per standard material:
          size  nunique          mean
material                             
BCR2G        4        2  56042.522500
BM90         2        1  15008.720000
GOR128       4        2  44597.335000
GOR132       4        1  60392.230000
KL2         17        3  77901.692941
N612         7        2  85048.698571

BCR-2G analyses:
013BCR2G    50900.00
014BCR2G    50900.00
028BCR2G    50900.00
070BCR2G    71470.09


**Ca43 is the internal standard** — constant to ~1e-7 relative within each repeated material while every other element scatters at percent level (KL2 = 77 901.7 ppm Ca ≡ 10.90 wt% CaO; NIST612 = 85 048.7 ppm ≡ 11.90 wt% CaO, the GeoReM value). Sample Ca varies per inclusion, so each melt inclusion was normalised to **its own EPMA CaO**.

Open question for 03: which EPMA CaO was used for the `MC22-…` inclusions? The glass session here is `UnknA-…`/`1982-…` and the olivine session is `UnkA-…MI`, so the LA samples come from a third measurement set we do not have. This is not a blocker — accuracy/precision work only needs the standards — but it means the EPMA-vs-LA 1:1 consistency plot in the plan has no sample overlap to plot. Ask F. van der Zwan whether the inclusion EPMA data exists, or drop that figure.

Two further deviations from the plan, both flagged rather than guessed:
* **No quartz / Si-only material** was run, so the ⁴⁵Sc ← ²⁹Si¹⁶O oxide-production correction cannot be derived from this session. Sc is also a minor concern for glass-hosted inclusions rather than olivine. Either report Sc with a caveat or check for a published ThO⁺/oxide rate for the session.
* **ATHO-G is in `Standards.xlsx` but was not measured** in this run; the control standards are BCR-2G, GOR128-G, GOR132-G, BM90/21-G and KL2-as-unknown.
**Flag for 03:** run **070BCR2G was normalised to 71 470 ppm Ca** while the other three BCR-2G analyses used 50 900 ppm (the GeoReM value). Every concentration in that analysis is therefore scaled by ~0.71 relative to its siblings — either a mislabelled analysis or a wrong internal-standard entry in GLITTER. Do not average it with the other BCR-2G points before resolving this.


## 6. Detection limits and below-MDL values

Counted, never zeroed. `_all` carries a per-analysis D.L.(ppm) for EPMA; GLITTER writes `<value` strings for LA.

In [20]:
OX2EL = {"Na2O":"Na","SO3":"S","FeO":"Fe","SiO2":"Si","Cl":"Cl","F":"F","K2O":"K","MnO":"Mn",
         "Al2O3":"Al","P2O5":"P","MgO":"Mg","CaO":"Ca","TiO2":"Ti","Cr2O3":"Cr","NiO":"Ni"}

for name, ox, ints in [("GLASS", gl_ox, gl_int), ("OLIVINE", ol_ox, ol_int)]:
    dl = ints.groupby("element")["dl_ppm"].mean()
    sd = ints.groupby("element")["sd_pct"].median()
    oxcols = [c for c in ox.columns if c in OX2EL]
    tbl = pd.DataFrame({"mean_DL_ppm": [dl.get(OX2EL[c], np.nan) for c in oxcols],
                        "median_SD_pct": [sd.get(OX2EL[c], np.nan) for c in oxcols],
                        "mean_wt_pct": [ox[c].mean() for c in oxcols]}, index=oxcols)
    tbl["mean_ppm"] = tbl.mean_wt_pct * 1e4
    tbl["x_above_DL"] = tbl.mean_ppm / tbl.mean_DL_ppm
    print(f"===== {name}"); print(tbl.round(2).to_string()); print()

print("LA — analyses below MDL per element (of 85):")
print(bdl.sum().loc[lambda s: s > 0].sort_values(ascending=False).to_string())

===== GLASS
       mean_DL_ppm  median_SD_pct  mean_wt_pct   mean_ppm  x_above_DL
Na2O        166.13           1.79         4.13   41326.98      248.76
SO3         111.70          29.28         0.16    1578.61       14.13
FeO         379.96           1.72        11.64  116441.10      306.45
SiO2        228.77           0.40        48.32  483233.33     2112.30
Cl           45.40           5.43         0.20    2030.04       44.71
F           371.35          38.43         0.13    1309.27        3.53
K2O          61.10           0.98         1.70   16967.19      277.72
MnO         283.11          18.16         0.20    2048.27        7.23
Al2O3       161.63           0.69        15.27  152716.19      944.86
P2O5        152.23           5.06         0.62    6176.70       40.58
MgO         165.89           0.96         4.58   45775.63      275.95
CaO          88.75           0.47        10.59  105940.49     1193.66
TiO2        126.93           0.97         3.07   30672.28      241.64

===== O

First look at what will and will not be reportable (quantified properly in 01/02/03):

* Glass: S, Cl, F, P are only a few × their detection limit and carry median S.D. of tens of %; `?` flags appear in `_all` where the count statistics failed. F in particular is near-noise.
* Olivine: Al₂O₃ and CaO sit close to their DLs — which is also where contamination from the surrounding glass shows up, so low values and outliers there need the beam-overlap check in 02, not just a DL check.
* LA: below-MDL hits concentrate in the low-Z standards (BM90, GOR) rather than in the samples.

## 7. Reference values

`Standards.xlsx` has 5 sheets: `major elements` (EPMA standards, wt% oxide), `trace elements` (GeoReM full tables), `Trace elements LA sorted` (element-symbol columns in µg/g — the one to join on for LA), and two raw GeoReM csv dumps for ATHO-G and BCR-2G.

In [21]:
maj = pd.read_excel(STD, "major elements")
maj.columns = [str(c).strip() for c in maj.columns]
maj[maj.columns[0]] = maj[maj.columns[0]].astype(str).str.strip()
print(maj.to_string(index=False))

latr = pd.read_excel(STD, "Trace elements LA sorted", header=0)
print("\nLA reference sheet — materials:", latr.iloc[:, 0].dropna().unique()[:12])

        Comment     SiO2      TiO2     Al2O3       FeO       MnO     MgO       CaO      Na2O     K2O  P2O5       Cl     F     S
      Std VGA99    50.94      4.06     12.49      13.3      0.15    5.08       9.3      2.66    0.82  0.38  0.02713   NaN 0.017
        Std VG2    50.81      1.85     14.06     11.84      0.22    6.71     11.12      2.62    0.19  0.20  0.03291 0.032 0.143
  Std Scapolite    49.78       NaN     25.05      0.17       NaN     NaN     13.58       5.2    0.94   NaN     1.43 0.016 0.530
      Std ALV98    49.53      1.27     16.58      8.42      0.14    8.68     11.81      2.88    0.05  0.06    0.009 0.013 0.113
        Std KE3      NaN       NaN       NaN       NaN       NaN     NaN       NaN       NaN     NaN   NaN     0.39 0.470 0.012
            NaN    SiO2     TiO2      Al2O3     FeO       MgO       CaO       Na2O    K2O      Total   NaN      NaN   NaN   NaN
Std Plagioclase    51.25      0.05     30.91      0.46      0.14   13.64      3.45      0.18  100.17   N

Note for 01: the `major elements` sheet has **no values for KE3** and no MnO/MgO/TiO₂ for Scapolite, so accuracy for F (calibrated on KE3) cannot be assessed against this file — say so rather than leaving a blank cell.
Values are also given as FeO, so any FeO/Fe₂O₃ conversion must be stated explicitly.

## 8. Save tidy frames

In [22]:
out = {"glass_oxide": gl_ox, "glass_meta": gl_meta, "glass_intensity": gl_int, "glass_calib": gl_cal,
       "ol_oxide": ol_ox, "ol_mole": ol_mol, "ol_ratios": ol_rat, "ol_meta": ol_meta,
       "ol_intensity": ol_int, "ol_calib": ol_cal, "la_index": la_idx}
for k, v in out.items():
    v.to_csv(INTERIM / f"{k}.csv", index=False)

SHORT = {"Trace Element Concentrations MDL filtered": "la_conc",
         "Trace Element Concentrations, Not filtered for MDL": "la_conc_unfiltered",
         "1 sigma error": "la_1sigma",
         "Minimum detection limits (99% confidence)": "la_mdl",
         "Trace element concentrations normalised to chondrite": "la_chondrite",
         "Mean Raw CPS background NOT subtracted": "la_cps_raw",
         "Mean Raw CPS background subtracted": "la_cps_bgsub"}
for long, short in SHORT.items():
    v, m = la[long]
    v.to_csv(INTERIM / f"{short}.csv")
    m.to_csv(INTERIM / f"{short}_bdl.csv")

print("\n".join(sorted(p.name for p in INTERIM.iterdir())))

glass_calib.csv
glass_intensity.csv
glass_meta.csv
glass_oxide.csv
la_1sigma.csv
la_1sigma_bdl.csv
la_chondrite.csv
la_chondrite_bdl.csv
la_conc.csv
la_conc_bdl.csv
la_conc_unfiltered.csv
la_conc_unfiltered_bdl.csv
la_cps_bgsub.csv
la_cps_bgsub_bdl.csv
la_cps_raw.csv
la_cps_raw_bdl.csv
la_index.csv
la_mdl.csv
la_mdl_bdl.csv
ol_calib.csv
ol_intensity.csv
ol_meta.csv
ol_mole.csv
ol_oxide.csv
ol_ratios.csv


## Carry into 01 / 02 / 03

1. Glass accuracy rests on **VG-2**, not VGA99 (which is the calibration standard for 9 of 13 oxides). Olivine accuracy rests on CaO/MnO/NiO/Cr₂O₃/Al₂O₃, not on the SC-Ol majors.
2. Order drift plots by the `_all` **timestamp**, not by `No.`.
3. 15 SC-Ol analyses, not 3 — check whether they cluster in time before treating them as session brackets.
4. LA run 033 is missing; KL2 brackets give the batch boundaries for a batch-wise drift correction.
5. LA internal standard is Ca43; sample CaO came from an EPMA dataset not in these files → the EPMA-vs-LA 1:1 figure has no overlapping samples.
6. No quartz → no measured Si-oxide production factor for the Sc45 correction.
7. No reference values for KE3 → no accuracy number for F.